In [1]:
# ============================================================
# PS S6E5 - exp_20260504_010_blend_core_006b_007_008_009_min
#
# Core blend stage:
# - 006b CatBoost original prior no Year
# - 007 RealMLP shared001 min
# - 008 RealMLP + safe orig prior
# - 009 LGB shared003 full-FE single
#
# Compare:
# - Avg
# - Ridge
# - ElasticNet
# - LogisticRegression
# - Hill Climb
# - Nelder-Mead
#
# Purpose:
# - Check whether 009 receives useful weight
# - Decide whether 009 + orig prior should be tried next
# ============================================================

In [2]:
import os
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.optimize import minimize
from scipy.stats import rankdata, spearmanr

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, ElasticNet, LogisticRegression

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

In [3]:
# ============================================================
# Config
# ============================================================

class CFG:
    EXP_ID = "exp_20260504_010_blend_core_006b_007_008_009_min"
    COMPETITION = "PS S6E5 Predicting F1 Pit Stops"
    METRIC = "AUC"

    COMP_BASE = Path("/kaggle/input/competitions/playground-series-s6e5")
    TRAIN_PATH = COMP_BASE / "train.csv"
    SUB_PATH = COMP_BASE / "sample_submission.csv"

    NPY_BASE = Path("/kaggle/input/datasets/mizushimatoshihiko/npy-files")

    OUTDIR = Path(f"/kaggle/working/{EXP_ID}")

    ID_COL = "id"
    TARGET = "PitNextLap"

    SEED = 42
    N_SPLITS = 5

    EPS = 1e-6


CFG.OUTDIR.mkdir(parents=True, exist_ok=True)

In [4]:
# ============================================================
# Utilities
# ============================================================

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


def safe_clip(x, eps=CFG.EPS):
    return np.clip(np.asarray(x, dtype=float), eps, 1.0 - eps)


def safe_logit(p, eps=CFG.EPS):
    p = safe_clip(p, eps)
    return np.log(p / (1.0 - p))


def rank01(x):
    x = np.asarray(x, dtype=float)
    return rankdata(x, method="average") / (len(x) + 1.0)


def softmax(z):
    z = np.asarray(z, dtype=float)
    z = z - np.max(z)
    e = np.exp(z)
    return e / e.sum()


def normalize_weights(w):
    w = np.asarray(w, dtype=float)
    w = np.clip(w, 0.0, None)
    s = w.sum()
    if s <= 0:
        return np.ones_like(w) / len(w)
    return w / s


def weighted_average(X, w):
    w = normalize_weights(w)
    return np.asarray(X) @ w


def auc(y, pred):
    return roc_auc_score(y, pred)


def sanitize_name(name: str) -> str:
    return (
        name.replace(" ", "_")
            .replace("/", "_")
            .replace("+", "plus")
            .replace("-", "_")
            .replace(".", "_")
    )


seed_everything(CFG.SEED)

In [5]:
# ============================================================
# Load data
# ============================================================

train = pd.read_csv(CFG.TRAIN_PATH)
sub = pd.read_csv(CFG.SUB_PATH)

y = train[CFG.TARGET].astype(int).values

print("train:", train.shape)
print("sub  :", sub.shape)
print("target mean:", y.mean())

train: (439140, 16)
sub  : (188165, 2)
target mean: 0.19898210137996994


In [6]:
# ============================================================
# Load OOF / pred
# ============================================================

MODEL_SPECS = [
    {
        "key": "006b",
        "name": "006b_cat_orig_prior_no_year",
        "family": "CatBoost",
        "oof": "oof_exp_20260503_006b_cat_original_prior_no_year_prior_min.npy",
        "pred": "pred_exp_20260503_006b_cat_original_prior_no_year_prior_min.npy",
        "public_lb": 0.94840,
    },
    {
        "key": "007",
        "name": "007_realmlp_shared001_min",
        "family": "RealMLP",
        "oof": "oof_exp_20260503_007_realmlp_shared001_min.npy",
        "pred": "pred_exp_20260503_007_realmlp_shared001_min.npy",
        "public_lb": 0.95273,
    },
    {
        "key": "008",
        "name": "008_realmlp_plus_orig_prior",
        "family": "RealMLP",
        "oof": "oof_exp_20260503_008_realmlp_shared001_plus_006b_orig_prior.npy",
        "pred": "pred_exp_20260503_008_realmlp_shared001_plus_006b_orig_prior.npy",
        "public_lb": 0.95249,
    },
    # {
    #     "key": "009",
    #     "name": "009_lgb_shared003_full_fe_single",
    #     "family": "LightGBM",
    #     "oof": "oof_exp_20260503_009_lgb_shared003_full_fe_single_min.npy",
    #     "pred": "pred_exp_20260503_009_lgb_shared003_full_fe_single_min.npy",
    #     "public_lb": 0.95076,
    # },
    {
        "key": "011",
        "name": "lgb_shared003_full_fe_plus_orig_prior_min",
        "family": "LightGBM",
        "oof": "oof_exp_20260504_011_lgb_shared003_full_fe_plus_orig_prior_min.npy",
        "pred": "pred_exp_20260504_011_lgb_shared003_full_fe_plus_orig_prior_min.npy",
        "public_lb": 0.93877,
    },
]

oofs = {}
preds = {}

for spec in MODEL_SPECS:
    oof_path = CFG.NPY_BASE / spec["oof"]
    pred_path = CFG.NPY_BASE / spec["pred"]

    assert oof_path.exists(), f"missing oof: {oof_path}"
    assert pred_path.exists(), f"missing pred: {pred_path}"

    oof = np.load(oof_path)
    pred = np.load(pred_path)

    assert len(oof) == len(train), (spec["key"], len(oof), len(train))
    assert len(pred) == len(sub), (spec["key"], len(pred), len(sub))
    assert np.isfinite(oof).all(), spec["key"]
    assert np.isfinite(pred).all(), spec["key"]

    oofs[spec["key"]] = oof.astype(float)
    preds[spec["key"]] = pred.astype(float)

model_keys = [s["key"] for s in MODEL_SPECS]
model_names = [s["name"] for s in MODEL_SPECS]

X_raw = np.column_stack([oofs[k] for k in model_keys])
T_raw = np.column_stack([preds[k] for k in model_keys])

print("X_raw:", X_raw.shape)
print("T_raw:", T_raw.shape)

X_raw: (439140, 4)
T_raw: (188165, 4)


In [7]:
# ============================================================
# Individual reports
# ============================================================

individual_rows = []

for i, spec in enumerate(MODEL_SPECS):
    pred_oof = X_raw[:, i]
    individual_rows.append({
        "key": spec["key"],
        "name": spec["name"],
        "family": spec["family"],
        "cv_auc": auc(y, pred_oof),
        "public_lb": spec["public_lb"],
        "oof_min": float(pred_oof.min()),
        "oof_max": float(pred_oof.max()),
        "pred_min": float(T_raw[:, i].min()),
        "pred_max": float(T_raw[:, i].max()),
    })

individual_df = pd.DataFrame(individual_rows).sort_values("cv_auc", ascending=False)
display(individual_df)

best_single_key = individual_df.iloc[0]["key"]
best_single_auc = individual_df.iloc[0]["cv_auc"]

print("best single:", best_single_key, best_single_auc)

,key,name,family,cv_auc,public_lb,oof_min,oof_max,pred_min,pred_max
1,007,007_realmlp_shared001_min,RealMLP,0.953270,0.95273,1.565386e-05,0.998289,2.742153e-05,0.997671
2,008,008_realmlp_plus_orig_prior,RealMLP,0.953039,0.95249,8.181750e-06,0.999127,1.578388e-05,0.998304
0,006b,006b_cat_orig_prior_no_year,CatBoost,0.948794,0.94840,2.389291e-05,0.996329,4.519774e-05,0.992921
3,011,lgb_shared003_full_fe_plus_orig_prior_min,LightGBM,0.936856,0.93877,3.320203e-07,0.999391,3.229298e-07,0.998751


best single: 007 0.9532701289015783


In [8]:
# ============================================================
# Correlation matrix
# ============================================================

pearson = pd.DataFrame(
    np.corrcoef(X_raw.T),
    index=model_keys,
    columns=model_keys,
)

spearman_mat = np.zeros((len(model_keys), len(model_keys)))

for i in range(len(model_keys)):
    for j in range(len(model_keys)):
        spearman_mat[i, j] = spearmanr(X_raw[:, i], X_raw[:, j]).correlation

spearman_df = pd.DataFrame(
    spearman_mat,
    index=model_keys,
    columns=model_keys,
)

print("Pearson correlation")
display(pearson)

print("Spearman correlation")
display(spearman_df)

pearson.to_csv(CFG.OUTDIR / "corr_pearson.csv")
spearman_df.to_csv(CFG.OUTDIR / "corr_spearman.csv")

Pearson correlation


,006b,007,008,011
006b,1.000000,0.974165,0.976387,0.934430
007,0.974165,1.000000,0.994703,0.935219
008,0.976387,0.994703,1.000000,0.936633
011,0.934430,0.935219,0.936633,1.000000


Spearman correlation


,006b,007,008,011
006b,1.000000,0.963871,0.964499,0.922721
007,0.963871,1.000000,0.991720,0.926116
008,0.964499,0.991720,1.000000,0.926787
011,0.922721,0.926116,0.926787,1.000000


In [9]:
# ============================================================
# Meta feature builders
# ============================================================

def build_meta_features(X, T):
    X = np.asarray(X, dtype=float)
    T = np.asarray(T, dtype=float)

    X_rank = np.column_stack([rank01(X[:, i]) for i in range(X.shape[1])])
    T_rank = np.column_stack([rank01(T[:, i]) for i in range(T.shape[1])])

    X_logit = np.column_stack([safe_logit(X[:, i]) for i in range(X.shape[1])])
    T_logit = np.column_stack([safe_logit(T[:, i]) for i in range(T.shape[1])])

    X_meta = np.column_stack([X, X_rank, X_logit])
    T_meta = np.column_stack([T, T_rank, T_logit])

    feature_names = (
        [f"raw_{k}" for k in model_keys] +
        [f"rank_{k}" for k in model_keys] +
        [f"logit_{k}" for k in model_keys]
    )

    return X_meta, T_meta, feature_names


X_meta, T_meta, meta_feature_names = build_meta_features(X_raw, T_raw)

print("X_meta:", X_meta.shape)
print("T_meta:", T_meta.shape)
print("meta features:", meta_feature_names)

X_meta: (439140, 12)
T_meta: (188165, 12)
meta features: ['raw_006b', 'raw_007', 'raw_008', 'raw_011', 'rank_006b', 'rank_007', 'rank_008', 'rank_011', 'logit_006b', 'logit_007', 'logit_008', 'logit_011']


In [10]:
# ============================================================
# Save candidate helper
# ============================================================

candidate_records = {}
candidate_summary = []

def add_candidate(
    name,
    method,
    oof_pred,
    test_pred,
    weights=None,
    params=None,
    notes=None,
):
    oof_pred = np.asarray(oof_pred, dtype=float)
    test_pred = np.asarray(test_pred, dtype=float)

    score = auc(y, oof_pred)

    candidate_records[name] = {
        "method": method,
        "oof": oof_pred,
        "pred": test_pred,
        "score": float(score),
        "weights": None if weights is None else [float(x) for x in weights],
        "params": params or {},
        "notes": notes or "",
    }

    candidate_summary.append({
        "name": name,
        "method": method,
        "cv_auc": float(score),
        "delta_vs_007": float(score - auc(y, oofs["007"])),
        "delta_vs_best_single": float(score - best_single_auc),
        "weights": None if weights is None else json.dumps([round(float(x), 8) for x in weights]),
        "params": json.dumps(params or {}, ensure_ascii=False),
        "notes": notes or "",
        "oof_min": float(oof_pred.min()),
        "oof_max": float(oof_pred.max()),
        "pred_min": float(test_pred.min()),
        "pred_max": float(test_pred.max()),
    })

    print(f"{name}: {score:.9f}")

In [11]:
# ============================================================
# Simple averages
# ============================================================

print("\n" + "=" * 80)
print("Simple averages")
print("=" * 80)

avg_specs = {
    # "avg_007_008": ["007", "008"],
    # "avg_007_009": ["007", "009"],
    # "avg_008_009": ["008", "009"],
    "avg_007_011": ["007", "011"],
    "avg_008_011": ["008", "011"],
    # "avg_006b_007": ["006b", "007"],
    # "avg_007_008_009": ["007", "008", "009"],
    # "avg_006b_007_009": ["006b", "007", "009"],
    "avg_007_008_011": ["007", "008", "011"],
    "avg_006b_007_011": ["006b", "007", "011"],
    # "avg_006b_007_008_009": ["006b", "007", "008", "009"],
    "avg_006b_007_008_011": ["006b", "007", "008", "011"],
}

for name, keys in avg_specs.items():
    idx = [model_keys.index(k) for k in keys]
    w = np.zeros(len(model_keys))
    for j in idx:
        w[j] = 1.0 / len(idx)

    add_candidate(
        name=name,
        method="avg",
        oof_pred=weighted_average(X_raw, w),
        test_pred=weighted_average(T_raw, w),
        weights=w,
        notes=f"simple average of {keys}",
    )


Simple averages
avg_007_011: 0.951042533
avg_008_011: 0.950868458
avg_007_008_011: 0.952618654
avg_006b_007_011: 0.952020207
avg_006b_007_008_011: 0.952902555


In [12]:
# ============================================================
# Ridge / ElasticNet / LogisticRegression meta CV
# Two-stage search
# ============================================================

meta_folds = list(
    StratifiedKFold(
        n_splits=CFG.N_SPLITS,
        shuffle=True,
        random_state=CFG.SEED,
    ).split(X_meta, y)
)


def make_refined_grid_1d(best_value, factor_low=0.25, factor_high=4.0, n=17, min_value=1e-8):
    lo = max(best_value * factor_low, min_value)
    hi = max(best_value * factor_high, lo * 1.01)
    return np.geomspace(lo, hi, n).tolist()


def run_meta_regressor_cv(estimator_factory, params):
    oof_meta = np.zeros(len(y), dtype=float)

    for tr_idx, va_idx in meta_folds:
        model = make_pipeline(
            StandardScaler(),
            estimator_factory(params),
        )
        model.fit(X_meta[tr_idx], y[tr_idx])
        oof_meta[va_idx] = model.predict(X_meta[va_idx])

    score = auc(y, oof_meta)
    return score, oof_meta


def run_meta_regressor_two_stage(name, estimator_factory, coarse_grid, refine_builder):
    history = []

    best = None

    print(f"\n{name} coarse search")
    for params in coarse_grid:
        score, oof_meta = run_meta_regressor_cv(estimator_factory, params)
        history.append({"stage": "coarse", "score": float(score), **params})
        print(params, score)

        if best is None or score > best["score"]:
            best = {
                "score": score,
                "params": params,
                "oof": oof_meta,
            }

    refined_grid = refine_builder(best["params"])

    print(f"\n{name} refined search around {best['params']}")
    for params in refined_grid:
        score, oof_meta = run_meta_regressor_cv(estimator_factory, params)
        history.append({"stage": "refined", "score": float(score), **params})
        print(params, score)

        if score > best["score"]:
            best = {
                "score": score,
                "params": params,
                "oof": oof_meta,
            }

    final_model = make_pipeline(
        StandardScaler(),
        estimator_factory(best["params"]),
    )
    final_model.fit(X_meta, y)
    test_meta = final_model.predict(T_meta)

    add_candidate(
        name=name,
        method=name.split("_")[0],
        oof_pred=best["oof"],
        test_pred=test_meta,
        params=best["params"],
        notes="two-stage meta CV on raw+rank+logit features",
    )

    hist_df = pd.DataFrame(history).sort_values("score", ascending=False)
    hist_df.to_csv(CFG.OUTDIR / f"{name}_search_history.csv", index=False)

    print(f"\n{name} best:", best["params"], best["score"])
    display(hist_df.head(30))

    return best, hist_df


def run_meta_logreg_cv(params):
    oof_meta = np.zeros(len(y), dtype=float)

    for tr_idx, va_idx in meta_folds:
        model = make_pipeline(
            StandardScaler(),
            LogisticRegression(
                C=params["C"],
                penalty="l2",
                solver="lbfgs",
                max_iter=3000,
                random_state=CFG.SEED,
            ),
        )
        model.fit(X_meta[tr_idx], y[tr_idx])
        oof_meta[va_idx] = model.predict_proba(X_meta[va_idx])[:, 1]

    score = auc(y, oof_meta)
    return score, oof_meta


def run_meta_logreg_two_stage(name, coarse_grid, refine_builder):
    history = []
    best = None

    print(f"\n{name} coarse search")
    for params in coarse_grid:
        score, oof_meta = run_meta_logreg_cv(params)
        history.append({"stage": "coarse", "score": float(score), **params})
        print(params, score)

        if best is None or score > best["score"]:
            best = {
                "score": score,
                "params": params,
                "oof": oof_meta,
            }

    refined_grid = refine_builder(best["params"])

    print(f"\n{name} refined search around {best['params']}")
    for params in refined_grid:
        score, oof_meta = run_meta_logreg_cv(params)
        history.append({"stage": "refined", "score": float(score), **params})
        print(params, score)

        if score > best["score"]:
            best = {
                "score": score,
                "params": params,
                "oof": oof_meta,
            }

    final_model = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=best["params"]["C"],
            penalty="l2",
            solver="lbfgs",
            max_iter=3000,
            random_state=CFG.SEED,
        ),
    )
    final_model.fit(X_meta, y)
    test_meta = final_model.predict_proba(T_meta)[:, 1]

    add_candidate(
        name=name,
        method="logreg",
        oof_pred=best["oof"],
        test_pred=test_meta,
        params=best["params"],
        notes="two-stage meta CV logistic regression on raw+rank+logit features",
    )

    hist_df = pd.DataFrame(history).sort_values("score", ascending=False)
    hist_df.to_csv(CFG.OUTDIR / f"{name}_search_history.csv", index=False)

    print(f"\n{name} best:", best["params"], best["score"])
    display(hist_df.head(30))

    return best, hist_df


print("\n" + "=" * 80)
print("Ridge / ElasticNet / LogReg two-stage search")
print("=" * 80)


# ----------------------------
# Ridge two-stage
# ----------------------------

ridge_coarse_grid = [
    {"alpha": a}
    for a in np.geomspace(1e-3, 1e3, 19)
]

ridge_best, ridge_hist = run_meta_regressor_two_stage(
    name="ridge_meta_all",
    estimator_factory=lambda p: Ridge(
        alpha=p["alpha"],
        random_state=CFG.SEED,
    ),
    coarse_grid=ridge_coarse_grid,
    refine_builder=lambda best_p: [
        {"alpha": a}
        for a in make_refined_grid_1d(
            best_p["alpha"],
            factor_low=0.2,
            factor_high=5.0,
            n=21,
            min_value=1e-6,
        )
    ],
)


# ----------------------------
# ElasticNet two-stage
# ----------------------------

elastic_coarse_grid = [
    {"alpha": a, "l1_ratio": l1}
    for a in np.geomspace(1e-5, 1e-1, 13)
    for l1 in [0.02, 0.05, 0.10, 0.20, 0.35, 0.50, 0.70, 0.90]
]


def build_elastic_refined_grid(best_p):
    alpha_grid = make_refined_grid_1d(
        best_p["alpha"],
        factor_low=0.25,
        factor_high=4.0,
        n=17,
        min_value=1e-7,
    )

    l1_center = best_p["l1_ratio"]
    l1_grid = np.linspace(
        max(0.001, l1_center - 0.20),
        min(0.999, l1_center + 0.20),
        9,
    ).tolist()

    # 境界付近だった場合の保険
    l1_grid = sorted(set(
        [round(x, 6) for x in l1_grid] +
        [0.01, 0.02, 0.05, 0.10, 0.30, 0.50, 0.70, 0.90]
    ))

    return [
        {"alpha": a, "l1_ratio": l1}
        for a in alpha_grid
        for l1 in l1_grid
    ]


elastic_best, elastic_hist = run_meta_regressor_two_stage(
    name="elasticnet_meta_all",
    estimator_factory=lambda p: ElasticNet(
        alpha=p["alpha"],
        l1_ratio=p["l1_ratio"],
        max_iter=30000,
        random_state=CFG.SEED,
        selection="cyclic",
    ),
    coarse_grid=elastic_coarse_grid,
    refine_builder=build_elastic_refined_grid,
)


# ----------------------------
# LogisticRegression two-stage
# ----------------------------

logreg_coarse_grid = [
    {"C": c}
    for c in np.geomspace(1e-3, 1e3, 19)
]

logreg_best, logreg_hist = run_meta_logreg_two_stage(
    name="logreg_meta_all",
    coarse_grid=logreg_coarse_grid,
    refine_builder=lambda best_p: [
        {"C": c}
        for c in make_refined_grid_1d(
            best_p["C"],
            factor_low=0.2,
            factor_high=5.0,
            n=21,
            min_value=1e-6,
        )
    ],
)


Ridge / ElasticNet / LogReg two-stage search

ridge_meta_all coarse search
{'alpha': np.float64(0.001)} 0.9535754010415377
{'alpha': np.float64(0.0021544346900318843)} 0.9535754011716739
{'alpha': np.float64(0.004641588833612777)} 0.9535754014970141
{'alpha': np.float64(0.01)} 0.9535754016922182
{'alpha': np.float64(0.021544346900318832)} 0.9535754024730347
{'alpha': np.float64(0.046415888336127774)} 0.9535754052058927
{'alpha': np.float64(0.1)} 0.9535754077435465
{'alpha': np.float64(0.21544346900318823)} 0.9535754129164562
{'alpha': np.float64(0.46415888336127775)} 0.9535754233598777
{'alpha': np.float64(1.0)} 0.9535754433357679
{'alpha': np.float64(2.154434690031882)} 0.9535754860529403
{'alpha': np.float64(4.6415888336127775)} 0.9535755902919505
{'alpha': np.float64(10.0)} 0.9535758282132619
{'alpha': np.float64(21.54434690031882)} 0.953576386074157
{'alpha': np.float64(46.41588833612773)} 0.9535775938672327
{'alpha': np.float64(100.0)} 0.9535802945490294
{'alpha': np.float64(215.

,stage,score,alpha
30,refined,0.953591,545.209817
17,coarse,0.953591,464.158883
29,refined,0.953591,464.158883
31,refined,0.953590,640.413779
28,refined,0.953590,395.156988
27,refined,0.953589,336.412919
32,refined,0.953588,752.242156
26,refined,0.953587,286.401749
25,refined,0.953586,243.825243
16,coarse,0.953585,215.443469



elasticnet_meta_all coarse search
{'alpha': np.float64(1e-05), 'l1_ratio': 0.02} 0.9535758572336105
{'alpha': np.float64(1e-05), 'l1_ratio': 0.05} 0.9535763090661242
{'alpha': np.float64(1e-05), 'l1_ratio': 0.1} 0.9535770120938307
{'alpha': np.float64(1e-05), 'l1_ratio': 0.2} 0.9535783167406935
{'alpha': np.float64(1e-05), 'l1_ratio': 0.35} 0.953580100190775
{'alpha': np.float64(1e-05), 'l1_ratio': 0.5} 0.9535814997068492
{'alpha': np.float64(1e-05), 'l1_ratio': 0.7} 0.9535826782193042
{'alpha': np.float64(1e-05), 'l1_ratio': 0.9} 0.9535842955181296
{'alpha': np.float64(2.1544346900318823e-05), 'l1_ratio': 0.02} 0.9535763727026736
{'alpha': np.float64(2.1544346900318823e-05), 'l1_ratio': 0.05} 0.9535772579859776
{'alpha': np.float64(2.1544346900318823e-05), 'l1_ratio': 0.1} 0.9535786442281705
{'alpha': np.float64(2.1544346900318823e-05), 'l1_ratio': 0.2} 0.9535809968284533
{'alpha': np.float64(2.1544346900318823e-05), 'l1_ratio': 0.35} 0.9535829366045162
{'alpha': np.float64(2.1544346

,stage,score,alpha,l1_ratio
269,refined,0.953713,0.006564,0.075750
270,refined,0.953712,0.006564,0.100000
253,refined,0.953712,0.005520,0.075750
271,refined,0.953712,0.006564,0.113125
284,refined,0.953712,0.007806,0.050000
285,refined,0.953712,0.007806,0.075750
286,refined,0.953712,0.007806,0.100000
287,refined,0.953712,0.007806,0.113125
254,refined,0.953712,0.005520,0.100000
272,refined,0.953712,0.006564,0.150500



logreg_meta_all coarse search
{'C': np.float64(0.001)} 0.9535938604233365
{'C': np.float64(0.0021544346900318843)} 0.9536187188570826
{'C': np.float64(0.004641588833612777)} 0.9536290440824109
{'C': np.float64(0.01)} 0.953629229721547
{'C': np.float64(0.021544346900318832)} 0.953626303611504
{'C': np.float64(0.046415888336127774)} 0.9536273636675807
{'C': np.float64(0.1)} 0.9536236508197896
{'C': np.float64(0.21544346900318823)} 0.9536276332119625
{'C': np.float64(0.46415888336127775)} 0.953627199793706
{'C': np.float64(1.0)} 0.9536275363907096
{'C': np.float64(2.154434690031882)} 0.9536270428495782
{'C': np.float64(4.6415888336127775)} 0.9536271700901429
{'C': np.float64(10.0)} 0.9536271905540434
{'C': np.float64(21.54434690031882)} 0.9536273263835902
{'C': np.float64(46.41588833612773)} 0.9536273283356316
{'C': np.float64(100.0)} 0.9536273254075694
{'C': np.float64(215.44346900318823)} 0.9536273208528062
{'C': np.float64(464.1588833612773)} 0.9536273240086064
{'C': np.float64(1000.0

,stage,score,C
33,refined,0.953631,0.019037
32,refined,0.953630,0.016207
28,refined,0.953630,0.008513
27,refined,0.953630,0.007248
26,refined,0.953630,0.006170
25,refined,0.953630,0.005253
31,refined,0.953630,0.013797
3,coarse,0.953629,0.010000
29,refined,0.953629,0.010000
2,coarse,0.953629,0.004642


In [13]:
# ============================================================
# Hill Climb non-negative weights
# ============================================================

print("\n" + "=" * 80)
print("Hill Climb")
print("=" * 80)

def hill_climb_weights(X, y, init_candidates=None):
    n = X.shape[1]

    candidates = []

    # one-hot starts
    for i in range(n):
        w = np.zeros(n)
        w[i] = 1.0
        candidates.append(w)

    # avg start
    candidates.append(np.ones(n) / n)

    if init_candidates:
        for w in init_candidates:
            candidates.append(normalize_weights(w))

    best_w = None
    best_score = -np.inf

    for w in candidates:
        s = auc(y, weighted_average(X, w))
        if s > best_score:
            best_score = s
            best_w = normalize_weights(w)

    for step in [0.20, 0.10, 0.05, 0.02, 0.01, 0.005, 0.002, 0.001]:
        improved = True

        while improved:
            improved = False

            for i in range(n):
                for direction in [-1, 1]:
                    trial = best_w.copy()
                    trial[i] += direction * step
                    trial = normalize_weights(trial)

                    s = auc(y, weighted_average(X, trial))

                    if s > best_score + 1e-12:
                        best_score = s
                        best_w = trial
                        improved = True

    return best_w, best_score


hc_init = []

# Use candidates that already looked good as starts
for cand_name in ["avg_007_008_009", "avg_006b_007_008_009"]:
    if cand_name in candidate_records:
        hc_init.append(candidate_records[cand_name]["weights"])

hc_w, hc_score = hill_climb_weights(X_raw, y, init_candidates=hc_init)

add_candidate(
    name="hc_nonnegative_raw",
    method="hc",
    oof_pred=weighted_average(X_raw, hc_w),
    test_pred=weighted_average(T_raw, hc_w),
    weights=hc_w,
    params={"constraint": "nonnegative_sum1"},
    notes="greedy hill climb on raw OOF predictions; in-sample optimizer, interpret cautiously",
)

print("HC weights:")
for k, w in zip(model_keys, hc_w):
    print(k, w)


Hill Climb
hc_nonnegative_raw: 0.953736573
HC weights:
006b 0.11597811341623827
007 0.51476830998503
008 0.3076773423316656
011 0.061576234267065995


In [14]:
# ============================================================
# Nelder-Mead softmax weights
# ============================================================

print("\n" + "=" * 80)
print("Nelder-Mead")
print("=" * 80)

def nm_optimize_weights(X, y, start_w=None, maxiter=500):
    n = X.shape[1]

    if start_w is None:
        start_w = np.ones(n) / n

    start_w = normalize_weights(start_w)
    z0 = np.log(np.clip(start_w, 1e-8, 1.0))

    def objective(z):
        w = softmax(z)
        p = weighted_average(X, w)
        return -auc(y, p)

    res = minimize(
        objective,
        z0,
        method="Nelder-Mead",
        options={
            "maxiter": maxiter,
            "xatol": 1e-7,
            "fatol": 1e-10,
            "disp": True,
        },
    )

    w = softmax(res.x)
    score = auc(y, weighted_average(X, w))

    return w, score, res


nm_w, nm_score, nm_res = nm_optimize_weights(
    X_raw,
    y,
    start_w=hc_w,
    maxiter=500,
)

add_candidate(
    name="nm_softmax_raw",
    method="nm",
    oof_pred=weighted_average(X_raw, nm_w),
    test_pred=weighted_average(T_raw, nm_w),
    weights=nm_w,
    params={
        "constraint": "softmax_sum1",
        "success": bool(nm_res.success),
        "message": str(nm_res.message),
        "nit": int(getattr(nm_res, "nit", -1)),
        "fun": float(nm_res.fun),
    },
    notes="Nelder-Mead on softmax weights; in-sample optimizer, interpret cautiously",
)

print("NM weights:")
for k, w in zip(model_keys, nm_w):
    print(k, w)


Nelder-Mead
Optimization terminated successfully.
         Current function value: -0.953737
         Iterations: 71
         Function evaluations: 177
nm_softmax_raw: 0.953736585
NM weights:
006b 0.11590242660702801
007 0.5145075258622679
008 0.30801682466786556
011 0.061573222862838464


In [15]:
# ============================================================
# Summary
# ============================================================

summary_df = pd.DataFrame(candidate_summary).sort_values("cv_auc", ascending=False)
display(summary_df)

summary_df.to_csv(CFG.OUTDIR / "blend_summary.csv", index=False)

# Expand weights table
weights_rows = []

for name, rec in candidate_records.items():
    if rec["weights"] is None:
        continue

    row = {
        "candidate": name,
        "method": rec["method"],
        "cv_auc": rec["score"],
    }

    for k, w in zip(model_keys, rec["weights"]):
        row[f"w_{k}"] = float(w)

    weights_rows.append(row)

weights_df = pd.DataFrame(weights_rows).sort_values("cv_auc", ascending=False)
display(weights_df)

weights_df.to_csv(CFG.OUTDIR / "blend_weights.csv", index=False)

,name,method,cv_auc,delta_vs_007,delta_vs_best_single,weights,params,notes,oof_min,oof_max,pred_min,pred_max
9,nm_softmax_raw,nm,0.953737,0.000466,0.000466,"[0.11590243, 0.51450753, 0.30801682, 0.06157322]","{""constraint"": ""softmax_sum1"", ""success"": true...",Nelder-Mead on softmax weights; in-sample opti...,0.000025,0.998091,0.000030,0.996943
8,hc_nonnegative_raw,hc,0.953737,0.000466,0.000466,"[0.11597811, 0.51476831, 0.30767734, 0.06157623]","{""constraint"": ""nonnegative_sum1""}",greedy hill climb on raw OOF predictions; in-s...,0.000025,0.998090,0.000030,0.996942
6,elasticnet_meta_all,elasticnet,0.953713,0.000442,0.000442,None,"{""alpha"": 0.006564197879454705, ""l1_ratio"": 0....",two-stage meta CV on raw+rank+logit features,-0.001775,0.991683,-0.001518,0.989017
7,logreg_meta_all,logreg,0.953631,0.000361,0.000361,None,"{""C"": 0.019036539387158793}",two-stage meta CV logistic regression on raw+r...,0.000034,0.992504,0.000040,0.989303
5,ridge_meta_all,ridge,0.953591,0.000321,0.000321,None,"{""alpha"": 545.2098169987383}",two-stage meta CV on raw+rank+logit features,-0.005593,0.988027,-0.005396,0.984283
4,avg_006b_007_008_011,avg,0.952903,-0.000368,-0.000368,"[0.25, 0.25, 0.25, 0.25]",{},"simple average of ['006b', '007', '008', '011']",0.000028,0.997882,0.000032,0.996194
2,avg_007_008_011,avg,0.952619,-0.000651,-0.000651,"[0.0, 0.33333333, 0.33333333, 0.33333333]",{},"simple average of ['007', '008', '011']",0.000012,0.998684,0.000017,0.998055
3,avg_006b_007_011,avg,0.952020,-0.001250,-0.001250,"[0.33333333, 0.33333333, 0.0, 0.33333333]",{},"simple average of ['006b', '007', '011']",0.000028,0.997467,0.000036,0.995675
0,avg_007_011,avg,0.951043,-0.002228,-0.002228,"[0.0, 0.5, 0.0, 0.5]",{},"simple average of ['007', '011']",0.000011,0.998463,0.000015,0.998208
1,avg_008_011,avg,0.950868,-0.002402,-0.002402,"[0.0, 0.0, 0.5, 0.5]",{},"simple average of ['008', '011']",0.000009,0.999049,0.000012,0.998247


,candidate,method,cv_auc,w_006b,w_007,w_008,w_011
6,nm_softmax_raw,nm,0.953737,0.115902,0.514508,0.308017,0.061573
5,hc_nonnegative_raw,hc,0.953737,0.115978,0.514768,0.307677,0.061576
4,avg_006b_007_008_011,avg,0.952903,0.250000,0.250000,0.250000,0.250000
2,avg_007_008_011,avg,0.952619,0.000000,0.333333,0.333333,0.333333
3,avg_006b_007_011,avg,0.952020,0.333333,0.333333,0.000000,0.333333
0,avg_007_011,avg,0.951043,0.000000,0.500000,0.000000,0.500000
1,avg_008_011,avg,0.950868,0.000000,0.000000,0.500000,0.500000


In [16]:
# ============================================================
# Save OOF / pred / submissions
# ============================================================

print("\n" + "=" * 80)
print("Save blend artifacts")
print("=" * 80)

submission_dir = CFG.OUTDIR / "submissions"
submission_dir.mkdir(parents=True, exist_ok=True)

target_col = [c for c in sub.columns if c != CFG.ID_COL][0]

for name, rec in candidate_records.items():
    safe_name = sanitize_name(name)

    oof_path = CFG.OUTDIR / f"oof_{CFG.EXP_ID}_{safe_name}.npy"
    pred_path = CFG.OUTDIR / f"pred_{CFG.EXP_ID}_{safe_name}.npy"
    sub_path = submission_dir / f"submission_{CFG.EXP_ID}_{safe_name}.csv"

    np.save(oof_path, rec["oof"])
    np.save(pred_path, rec["pred"])

    sub_out = sub.copy()
    sub_out[target_col] = safe_clip(rec["pred"])
    sub_out.to_csv(sub_path, index=False)

    print(name, rec["score"], sub_path)


Save blend artifacts
avg_007_011 0.9510425329898538 /kaggle/working/exp_20260504_010_blend_core_006b_007_008_009_min/submissions/submission_exp_20260504_010_blend_core_006b_007_008_009_min_avg_007_011.csv
avg_008_011 0.9508684584789097 /kaggle/working/exp_20260504_010_blend_core_006b_007_008_009_min/submissions/submission_exp_20260504_010_blend_core_006b_007_008_009_min_avg_008_011.csv
avg_007_008_011 0.9526186544780744 /kaggle/working/exp_20260504_010_blend_core_006b_007_008_009_min/submissions/submission_exp_20260504_010_blend_core_006b_007_008_009_min_avg_007_008_011.csv
avg_006b_007_011 0.9520202074215892 /kaggle/working/exp_20260504_010_blend_core_006b_007_008_009_min/submissions/submission_exp_20260504_010_blend_core_006b_007_008_009_min_avg_006b_007_011.csv
avg_006b_007_008_011 0.9529025546278117 /kaggle/working/exp_20260504_010_blend_core_006b_007_008_009_min/submissions/submission_exp_20260504_010_blend_core_006b_007_008_009_min_avg_006b_007_008_011.csv
ridge_meta_all 0.95359

In [17]:
# ============================================================
# Save result json
# ============================================================

result = {
    "experiment": {
        "id": CFG.EXP_ID,
        "competition": CFG.COMPETITION,
        "metric": CFG.METRIC,
        "created_at": "2026-05-04",
    },
    "inputs": {
        "model_specs": MODEL_SPECS,
        "model_keys": model_keys,
        "n_models": len(model_keys),
    },
    "individual": individual_df.to_dict(orient="records"),
    "correlation": {
        "pearson": pearson.to_dict(),
        "spearman": spearman_df.to_dict(),
    },
    "blend_summary": summary_df.to_dict(orient="records"),
    "blend_weights": weights_df.to_dict(orient="records"),
    "best": {
        "candidate": str(summary_df.iloc[0]["name"]),
        "method": str(summary_df.iloc[0]["method"]),
        "cv_auc": float(summary_df.iloc[0]["cv_auc"]),
    },
    "notes": [
        "This blend stage is intended to check whether 009 receives useful weight.",
        "Ridge/ElasticNet/LogReg use meta-CV on raw+rank+logit features.",
        "HC/NM optimize directly on full OOF and are more overfit-prone.",
        "If 009 receives meaningful weight, trying 009 + safe orig prior as next experiment is justified.",
        "If 009 receives little or zero weight, 009 + orig prior priority should be lowered.",
    ],
    "artifacts": {
        "outdir": str(CFG.OUTDIR),
        "blend_summary": str(CFG.OUTDIR / "blend_summary.csv"),
        "blend_weights": str(CFG.OUTDIR / "blend_weights.csv"),
        "submissions_dir": str(submission_dir),
    },
}

with open(CFG.OUTDIR / "blend_result.json", "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print("\nSaved to:", CFG.OUTDIR)
print("Best candidate:")
print(summary_df.iloc[0].to_dict())


Saved to: /kaggle/working/exp_20260504_010_blend_core_006b_007_008_009_min
Best candidate:
{'name': 'nm_softmax_raw', 'method': 'nm', 'cv_auc': 0.9537365849055583, 'delta_vs_007': 0.000466456003980098, 'delta_vs_best_single': 0.000466456003980098, 'weights': '[0.11590243, 0.51450753, 0.30801682, 0.06157322]', 'params': '{"constraint": "softmax_sum1", "success": true, "message": "Optimization terminated successfully.", "nit": 71, "fun": -0.9537365849055583}', 'notes': 'Nelder-Mead on softmax weights; in-sample optimizer, interpret cautiously', 'oof_min': 2.467574393684224e-05, 'oof_max': 0.9980908992143869, 'pred_min': 2.9757923746162184e-05, 'pred_max': 0.9969426001210643}
